# Triple-Barrier Label Selection — **Conservative** scheme, and an honest lenient-vs-conservative verdict

A stricter re-run of the model-free triple-barrier study
([`triple-barrier-label.ipynb`](triple-barrier-label.ipynb) / [`reports/jay/triple-barrier-label.md`](../../reports/jay/triple-barrier-label.md)).
That study's **own §4 caveats** flag its picks as *lenient / oracle-inflated*: 6 of 11 per-instrument
top-1 geometries sit at `pt=sl=0.25, h=1`, and **all 11** use the hair-trigger stop `sl=0.25`. Three
leniency sources motivate a stricter pass:

1. **`pt` too small** — at `pt=0.25` the profit barrier is touched almost trivially.
2. **`h=1` too short** — a 1-bar label overlaps the *same bar* as the lag-1 return it is scored against
   ⇒ the adjusted Sharpe is partly **circular** (self-fulfilling) in-sample.
3. **timeout = `sign(return)` too lenient** — a trade that merely drifts to the horizon and ends barely
   positive counts as a "win".

**This notebook operationalises all three, one mechanism each:**

| Lever | Lenient | **Conservative** | Fixes |
|---|---|---|---|
| `pt` (×σ) | `{0.25 … 2.5}` | **`{1, 1.5, 2, 2.5, 3}`** | leniency 1 |
| `sl` (×σ) | `{0.25 … 2.5}` | **`{0.5, 0.75, 1, 1.5, 2, 2.5}`** (drop hair-trigger `0.25`) | targets the tight-stop oracle bias (§8) |
| `h` (bars) | `{1,2,3,5,10,15,20}` | **`{5, 10, 15, 20}`** | leniency 2 (kills the h=1/lag-1 overlap) |
| timeout rule | `sign(ret)` | **`vertical_zero=True`** ⇒ timeout→`0` | leniency 3 |

⇒ **5×6×4 = 120 geometries**. Under `vertical_zero=True` a label is `y=1` **iff the profit barrier is
touched first** (PT-touch→1; SL-touch→0; timeout→0) — a strict, economically meaningful "win". Everything
else (per-instrument selection by **lag-1 adjusted-PnL Sharpe**, the 3-lag placebo-in-time, the 2022-H1
hold-out) is unchanged, so the two studies are directly comparable. We add a **degeneracy floor** (a strict
label can emit too few positives to be usable), a **lenient-vs-conservative comparison** (incl. a clean
`vertical_zero` ablation), and an **honest verdict**.

> Still an **oracle / in-sample** measure (realized `y` filters realized returns) — an upper bound on a
> learnable meta-model, not a backtest. The hold-out, the lag contrast, and the floor are the honesty guards.

## Section 0 — Setup

In [ ]:
%matplotlib inline
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42)

from stml.io import _find_repo_root
ROOT = _find_repo_root(Path.cwd().resolve()); RESULTS = ROOT / "results"
from stml.model.dataset import load_matrix, close_panel, attach_bar_pos, events_frame, DEV_PARTITIONS
from stml.model.evaluate import release_test, nav_sharpe
from stml.model.labels import triple_barrier_labels, load_instrument_geometry

matrix = attach_bar_pos(load_matrix(), close_panel())
close_wide = close_panel()
dev_matrix  = matrix[matrix.partition.isin(DEV_PARTITIONS)].reset_index(drop=True)
test_matrix = release_test(matrix, final_confirmation=True)   # 2022-H1 — a deliberate label-study hold-out
ev_dev, ev_test = events_frame(dev_matrix), events_frame(test_matrix)
INSTR = sorted(ev_dev.instrument.unique())
TRAIN_PRICE_END, TEST_PRICE_END = "2021-12-30", "2022-06-30"

# The lenient study's per-instrument geometry (tradeable lag-1 top-1) — for the apples-to-apples comparison.
LEN_GEOM = load_instrument_geometry(RESULTS / "triple_barrier_per_instrument.csv",
                                    pnl_type="lag1", rank_col="adj_sharpe_train")
print(f"dev events {len(ev_dev)} ({ev_dev.date.min().date()}->{ev_dev.date.max().date()}) | "
      f"test events {len(ev_test)} ({ev_test.date.min().date()}->{ev_test.date.max().date()}) | "
      f"instruments {len(INSTR)}")
print("lenient geometry loaded:", {k: LEN_GEOM[k] for k in INSTR})

## Section 0.5 — Per-event lagged signed returns (geometry-free; placebo baseline)

`g_i(L) = s_i · u_{t_i+L}` on each instrument's own calendar (entry-at-`t+1` ⇒ lag-1 is the first
tradeable bar). Identical to the lenient study and **independent of `(pt,sl,h)`**, so it is the common
baseline both arms share.

In [ ]:
def lagged_signed_returns(events, close_wide, lags=(0, 1, 2)):
    out = events[["date", "instrument", "side"]].copy().reset_index(drop=True)
    for L in lags:
        out[f"g{L}"] = np.nan
    for inst, g in out.groupby("instrument"):
        if inst not in close_wide.columns:
            continue
        s = close_wide[inst].dropna(); u = s.pct_change().to_numpy()
        pos = s.index.get_indexer(pd.DatetimeIndex(g["date"])); side = g["side"].to_numpy(float)
        for L in lags:
            vals = np.full(len(g), np.nan)
            ok = (pos >= 0) & (pos + L >= 0) & (pos + L < len(u))
            vals[ok] = u[pos[ok] + L] * side[ok]
            out.loc[g.index, f"g{L}"] = vals
    return out

G_dev, G_test = lagged_signed_returns(ev_dev, close_wide), lagged_signed_returns(ev_test, close_wide)

def raw_lag_table(G):
    rows = []
    for inst, g in G.groupby("instrument"):
        for L in (0, 1, 2):
            r = g[f"g{L}"].to_numpy(float); r = r[np.isfinite(r)]
            sh = nav_sharpe(pd.DataFrame({"ret": r}), np.ones(len(r), bool))["sharpe"]
            rows.append({"instrument": inst, "lag": L, "sharpe": round(sh, 3)})
    return pd.DataFrame(rows).pivot(index="instrument", columns="lag", values="sharpe")

print("RAW primary Sharpe by lag (no geometry, no filter) — placebo-in-time; expect lag 1 strongest:")
display(raw_lag_table(G_dev))

## Section 1 — The mechanic + what `vertical_zero` changes (worked example)

Same adjusted-signal mechanic as the lenient study (`adjusted = side · y`, keep only `y=1`). The **only**
new ingredient is the timeout rule. Below, one instrument/geometry labelled **both ways** — `vertical_zero
=False` (lenient: timeout→`sign(ret)`) vs `=True` (conservative: timeout→`0`) — so the flips are visible.
Every flip is a **timeout that ended barely positive**: the lenient rule banks it, the conservative rule
discards it (it never actually reached the profit target).

In [ ]:
inst0, (pt0, sl0, h0) = "es1s", (1.5, 1.0, 10)
e0 = ev_dev[ev_dev.instrument == inst0].reset_index(drop=True)
lab_len = triple_barrier_labels(close_wide, e0, pt=pt0, sl=sl0, h=h0, vertical_zero=False, price_end=TRAIN_PRICE_END)
lab_con = triple_barrier_labels(close_wide, e0, pt=pt0, sl=sl0, h=h0, vertical_zero=True,  price_end=TRAIN_PRICE_END)
cmp = (lab_len[["date", "touch", "ret", "bin"]].rename(columns={"bin": "y_lenient"})
       .merge(lab_con[["date", "bin"]].rename(columns={"bin": "y_conservative"}), on="date"))
flips = cmp[(cmp.touch == "vert") & (cmp.y_lenient == 1) & (cmp.y_conservative == 0)]
print(f"{inst0}  pt={pt0} sl={sl0} h={h0}:  {len(cmp)} events | "
      f"lenient pos-rate {cmp.y_lenient.mean():.3f} -> conservative {cmp.y_conservative.mean():.3f} | "
      f"timeout-positives demoted to 0: {len(flips)}")
print("\nExample flips (timeout exits the lenient rule banked, the conservative rule discards):")
display(flips.head(6).round(4))
# adjusted lag-1 equity under each rule
g0m = lab_len[["date"]].merge(G_dev[G_dev.instrument == inst0][["date", "g1"]], on="date").sort_values("date")
g0m["raw"] = g0m["g1"].astype(float)
g0m["adj_lenient"] = np.where(lab_len.sort_values("date")["bin"].to_numpy() == 1, g0m["raw"], 0.0)
g0m["adj_conserv"] = np.where(lab_con.sort_values("date")["bin"].to_numpy() == 1, g0m["raw"], 0.0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(g0m["date"], np.nancumsum(g0m["raw"]), label="raw primary", lw=1)
ax.plot(g0m["date"], np.nancumsum(g0m["adj_lenient"]), label="adjusted (lenient rule)", lw=1.2)
ax.plot(g0m["date"], np.nancumsum(g0m["adj_conserv"]), label="adjusted (conservative, vertical_zero)", lw=1.4)
ax.set_title(f"{inst0} pt={pt0} sl={sl0} h={h0} — lag-1 adjusted equity, lenient vs conservative timeout rule")
ax.legend(fontsize=8); ax.set_ylabel("cum. return"); plt.tight_layout(); plt.show()

## Section 2 — Why model-free, and why conservative

Each `(pt,sl,h, vertical_zero)` is a *different labeling* (different ground truth), so AUC is **not**
comparable across geometries — the labeling-invariant objective is the **Sharpe of the reconstructed
adjusted-PnL curve** (a good label separates winners from losers ⇒ the adjusted signal beats the raw
primary risk-adjusted). The conservative pass exists because the lenient study's own caveats (§4) name two
artifacts we now remove **structurally**: dropping `h<5` kills the *h=1/lag-1 single-bar overlap* (the
circular-inflation critique), and `vertical_zero=True` closes the *timeout-sign loophole*. Dropping the
hair-trigger `sl=0.25` is *meant* to defuse the *oracle-dominator* the lenient report flags (a tight stop
cuts losers to `y=0`, which the oracle rewards but which realises extra whipsaw live) — but §8 shows the
search simply re-pegs to the next-tightest stop (`sl=0.5`), so this **exposes** the oracle's structural
tight-stop preference rather than removing it. The 3-lag placebo-in-time
([`01-signal-direction.md`](../../reports/harry/01-signal-direction.md)) is retained: a *real* edge must
keep **lag-1** dominant over **lag-0** (contemporaneous, untradeable) and **lag-2** (decayed).

## Section 3 — Conservative grid, scoring, and the degeneracy floor

`score_all` labels once per `(pt,sl,h)` across all instruments (with `vertical_zero=True`), then scores
every `(instrument × pnl_type)`: `raw = nav_sharpe(all)`, `adj = nav_sharpe(y==1)`. `score_map` does the
same for a fixed per-instrument geometry dict (used for the lenient arm and the ablation). The **floor**
(`pos_rate ∈ [0.15, 0.85]` and `n_kept ≥ 15`, in-sample) rejects geometries whose strict label is too
one-sided / too thin to trust.

In [ ]:
CONS_PTS = [1.0, 1.5, 2.0, 2.5, 3.0]
CONS_SLS = [0.5, 0.75, 1.0, 1.5, 2.0, 2.5]
CONS_HS  = [5, 10, 15, 20]
CONS_GRID = [(pt, sl, h) for h in CONS_HS for pt in CONS_PTS for sl in CONS_SLS]
PNL_TYPES = [("exit", "ret"), ("lag0", "g0"), ("lag1", "g1"), ("lag2", "g2")]
POS_LO, POS_HI, N_KEPT_MIN = 0.15, 0.85, 15
print(f"conservative grid: {len(CONS_GRID)} geometries x {len(INSTR)} instruments x {len(PNL_TYPES)} pnl types"
      f"  |  floor: pos_rate in [{POS_LO},{POS_HI}] and n_kept >= {N_KEPT_MIN}")

def _rows_for(g, inst, pt, sl, h):
    y = (g["bin"] == 1).to_numpy()
    base = dict(instrument=inst, pt=pt, sl=sl, h=int(h), n=len(g), n_kept=int(y.sum()),
                pos_rate=round(float(g["bin"].mean()), 4),
                vert_frac=round(float((g["touch"] == "vert").mean()), 4))
    out = []
    for typ, col in PNL_TYPES:
        r = g[col].to_numpy(float); ok = np.isfinite(r)
        ev = pd.DataFrame({"ret": r[ok]}); ym = y[ok]
        raw = nav_sharpe(ev, np.ones(len(ev), bool)); adj = nav_sharpe(ev, ym)
        out.append({**base, "pnl_type": typ, "raw_sharpe": raw["sharpe"], "adj_sharpe": adj["sharpe"],
                    "adj_nav": adj["nav"], "n_taken": int(adj["n_taken"]),
                    "sharpe_lift": adj["sharpe"] - raw["sharpe"]})
    return out

def score_all(events, G, price_end, grid, vertical_zero):
    rows = []
    for pt, sl, h in grid:
        lab = triple_barrier_labels(close_wide, events, pt=pt, sl=sl, h=h,
                                    vertical_zero=vertical_zero, price_end=price_end)
        if lab.empty:
            continue
        mm = lab.merge(G, on=["date", "instrument"], how="left")
        for inst, g in mm.groupby("instrument"):
            rows += _rows_for(g, inst, pt, sl, h)
    return pd.DataFrame(rows)

def score_map(events, G, price_end, geom_map, vertical_zero):
    rows = []
    for inst, (pt, sl, h) in geom_map.items():
        e = events[events.instrument == inst]
        if e.empty:
            continue
        lab = triple_barrier_labels(close_wide, e, pt=pt, sl=sl, h=int(h),
                                    vertical_zero=vertical_zero, price_end=price_end)
        if lab.empty:
            continue
        mm = lab.merge(G, on=["date", "instrument"], how="left")
        for inst2, g in mm.groupby("instrument"):
            rows += _rows_for(g, inst2, pt, sl, int(h))
    return pd.DataFrame(rows)

def floor_pass(df):
    return (df.pos_rate >= POS_LO) & (df.pos_rate <= POS_HI) & (df.n_kept >= N_KEPT_MIN)

def select_top(df, lag, k=1, apply_floor=True):
    sub = df[df.pnl_type == lag].copy()
    if apply_floor:
        sub = sub[floor_pass(sub)]
    return (sub.sort_values(["instrument", "adj_sharpe", "h", "pt", "sl"],
                            ascending=[True, False, True, True, True])
               .groupby("instrument").head(k).reset_index(drop=True))

## Section 4 — In-sample grid search + floor (2020 → 2022-01-01)

In [ ]:
cons_train = score_all(ev_dev, G_dev, TRAIN_PRICE_END, CONS_GRID, vertical_zero=True)
print("conservative train rows:", len(cons_train))

cons_top1 = select_top(cons_train, "lag1", k=1, apply_floor=True)
passing = list(cons_top1.instrument)
floor_fail = [i for i in INSTR if i not in passing]
print(f"floor-passing instruments: {len(passing)}/{len(INSTR)}  ->", passing)
print("floor-FAIL (no usable conservative geometry):", floor_fail or "none")

print("\nConservative top-3 per FLOOR-PASSING instrument by in-sample lag-1 adjusted Sharpe:")
display(select_top(cons_train, "lag1", k=3, apply_floor=True)[
    ["instrument", "pt", "sl", "h", "pos_rate", "n_kept", "vert_frac", "raw_sharpe", "adj_sharpe"]].round(3))

# Why each floor-fail name fails: its best-adj geometry ignoring the floor, with pos_rate / n_kept.
if floor_fail:
    bf = (cons_train[(cons_train.pnl_type == "lag1") & (cons_train.instrument.isin(floor_fail))]
          .sort_values(["instrument", "adj_sharpe"], ascending=[True, False])
          .groupby("instrument").head(1))
    print("\nFloor-FAIL diagnostics — best-adj geometry (floor ignored) and why it is rejected:")
    display(bf[["instrument", "pt", "sl", "h", "pos_rate", "n_kept", "adj_sharpe"]].round(3)
            .assign(reason=np.where(bf.n_kept < N_KEPT_MIN, "n_kept<15",
                    np.where(bf.pos_rate < POS_LO, "pos_rate<0.15",
                    np.where(bf.pos_rate > POS_HI, "pos_rate>0.85", "ok")))))

# Floor sensitivity: how the passing set changes under a looser floor.
def n_pass(lo, hi, nk):
    s = cons_train[cons_train.pnl_type == "lag1"]
    s = s[(s.pos_rate >= lo) & (s.pos_rate <= hi) & (s.n_kept >= nk)]
    return s.instrument.nunique()
print(f"\nFloor sensitivity — instruments with >=1 eligible geometry:"
      f"  [0.15,0.85]&n>=15 -> {n_pass(0.15,0.85,15)} ;  [0.10,0.90]&n>=10 -> {n_pass(0.10,0.90,10)}")

## Section 5 — Hold-out validation (2022-H1)

For each floor-passing instrument's in-sample top-3 (lag-1), recompute Sharpe on the hold-out and report
its hold-out **rank among the 120 conservative geometries** (1 = best out-of-sample). The hold-out is
**not** floor-gated, so `n_taken_hold` is shown — a tiny taken-count makes the hold-out Sharpe noise.

In [ ]:
cons_hold_grid = score_all(ev_test, G_test, TEST_PRICE_END, CONS_GRID, vertical_zero=True)

def cons_train_vs_holdout(lag):
    tr = select_top(cons_train, lag, k=3, apply_floor=True)
    ho = cons_hold_grid[cons_hold_grid.pnl_type == lag].copy()
    ho["ho_rank"] = ho.groupby("instrument")["adj_sharpe"].rank(ascending=False).astype(int)
    return tr.merge(ho[["instrument", "pt", "sl", "h", "adj_sharpe", "raw_sharpe", "ho_rank", "n_taken"]],
                    on=["instrument", "pt", "sl", "h"], suffixes=("_train", "_hold"))

cons_tv = cons_train_vs_holdout("lag1")
print(f"Conservative top-3 (lag-1) in-sample vs HOLD-OUT — ho_rank out of {len(CONS_GRID)}:")
display(cons_tv[["instrument", "pt", "sl", "h", "adj_sharpe_train", "adj_sharpe_hold",
                 "raw_sharpe_hold", "ho_rank", "n_taken_hold"]].round(3))

In [ ]:
# Equity curves: each floor-passing instrument's in-sample #1 (lag-1) conservative geometry.
def equity(inst, pt, sl, h, col="g1", vertical_zero=True):
    parts = []
    for events, G, pe, part in [(ev_dev, G_dev, TRAIN_PRICE_END, "train"),
                                (ev_test, G_test, TEST_PRICE_END, "hold")]:
        e = events[events.instrument == inst]
        lab = triple_barrier_labels(close_wide, e, pt=pt, sl=sl, h=int(h),
                                    vertical_zero=vertical_zero, price_end=pe)
        mm = lab.merge(G, on=["date", "instrument"], how="left")
        parts.append(mm[["date", "bin", col]].assign(part=part))
    d = pd.concat(parts).sort_values("date").reset_index(drop=True)
    d["raw"] = d[col].astype(float); d["adj"] = np.where(d["bin"] == 1, d["raw"], 0.0)
    return d

cons_best1 = cons_top1.set_index("instrument")
ncol = 3; nrow = int(np.ceil(len(INSTR) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3.2 * nrow)); axes = axes.ravel()
for ax, inst in zip(axes, INSTR):
    if inst not in cons_best1.index:
        ax.text(0.5, 0.5, f"{inst}\nfloor-FAIL\n(no usable\nconservative label)",
                ha="center", va="center", fontsize=9, color="firebrick"); ax.axis("off"); continue
    r = cons_best1.loc[inst]; d = equity(inst, r.pt, r.sl, int(r.h), "g1")
    ax.plot(d["date"], np.nancumsum(d["raw"]), label="raw", lw=1)
    ax.plot(d["date"], np.nancumsum(d["adj"]), label="adjusted", lw=1.3)
    split = d.loc[d.part == "hold", "date"]
    if len(split):
        ax.axvline(split.iloc[0], c="k", ls=":", lw=0.8)
    ax.set_title(f"{inst}  pt={r.pt} sl={r.sl} h={int(r.h)} (lag-1, vz)", fontsize=9)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
for ax in axes[len(INSTR):]:
    ax.axis("off")
fig.suptitle("Conservative per-instrument #1 geometry — lag-1 raw vs adjusted (dotted = train/hold split)")
plt.tight_layout(); plt.show()

## Section 6 — Lag & stability diagnostics (conservative)

In [ ]:
# (a) Placebo-in-time: median (over floor-passing instruments) of in-sample top-1 adjusted Sharpe, by lag.
rows = []
for lag in ["lag0", "lag1", "lag2", "exit"]:
    b = select_top(cons_train, lag, k=1, apply_floor=True)
    rows.append({"pnl_type": lag, "n_instr": b.instrument.nunique(),
                 "median_top1_adj_sharpe": round(float(b.adj_sharpe.median()), 3)})
print("Conservative in-sample top-1 adjusted Sharpe by lag (median over floor-passing) — expect lag1 > lag0,lag2:")
display(pd.DataFrame(rows))

# (b) Does in-sample geometry choice generalise? train vs hold-out adj_sharpe across the 120, by lag.
from scipy.stats import spearmanr
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
for axi, lag in zip(ax, ["lag0", "lag1", "lag2"]):
    j = cons_train[cons_train.pnl_type == lag].merge(
        cons_hold_grid[cons_hold_grid.pnl_type == lag], on=["instrument", "pt", "sl", "h"],
        suffixes=("_tr", "_ho"))
    j = j.replace([np.inf, -np.inf], np.nan).dropna(subset=["adj_sharpe_tr", "adj_sharpe_ho"])
    rho, _ = spearmanr(j["adj_sharpe_tr"], j["adj_sharpe_ho"]) if len(j) > 2 else (np.nan, np.nan)
    axi.scatter(j["adj_sharpe_tr"], j["adj_sharpe_ho"], s=6, alpha=0.3)
    axi.axhline(0, c="k", lw=0.5); axi.axvline(0, c="k", lw=0.5)
    axi.set_title(f"{lag}: train vs hold-out adj Sharpe (Spearman {rho:.2f})")
    axi.set_xlabel("train adj Sharpe"); axi.set_ylabel("hold-out adj Sharpe")
plt.tight_layout(); plt.show()

## Section 7 — Conservative vs Lenient (apples-to-apples) + `vertical_zero` ablation

Both arms recomputed here under **identical** scoring (`score_map`, same `price_end`, same `nav_sharpe`).
Only the lenient *geometry* is read from the lenient study (`load_instrument_geometry`); every Sharpe is
recomputed. Three comparisons:

- **lenient** (its native rule, `vertical_zero=False`) vs **conservative** (floor-passing top-1,
  `vertical_zero=True`) on pos-rate, n_kept, vert-frac, lag-1 adj Sharpe (train + hold-out), raw hold-out;
- the **exit panel** (the realized first-touch return): the label is *defined from* this return, so it is
  the **maximally circular** panel — its Sharpe is the highest of all and, under `vertical_zero`, every kept
  exit ≈ `+pt·σ` by construction. We report it as the clearest illustration that these adjusted Sharpes are
  **oracle upper bounds**, *not* as evidence of edge;
- the **`vertical_zero` ablation**: the lenient geometry re-scored with *only* the timeout rule flipped —
  isolating the timeout-rule effect from the grid effect.

In [ ]:
# Lenient arm (native rule) and the vertical_zero ablation (lenient geometry, timeout rule flipped).
len_tr = score_map(ev_dev,  G_dev,  TRAIN_PRICE_END, LEN_GEOM, vertical_zero=False)
len_ho = score_map(ev_test, G_test, TEST_PRICE_END, LEN_GEOM, vertical_zero=False)
abl_tr = score_map(ev_dev,  G_dev,  TRAIN_PRICE_END, LEN_GEOM, vertical_zero=True)
# Conservative arm top-1 (re-scored train + hold-out for identical columns).
CONS_GEOM = {r.instrument: (float(r.pt), float(r.sl), int(r.h)) for r in cons_top1.itertuples()}
cons_tr = score_map(ev_dev,  G_dev,  TRAIN_PRICE_END, CONS_GEOM, vertical_zero=True)
cons_ho = score_map(ev_test, G_test, TEST_PRICE_END, CONS_GEOM, vertical_zero=True)

def l1(df): return df[df.pnl_type == "lag1"].set_index("instrument")
def ex(df): return df[df.pnl_type == "exit"].set_index("instrument")
Ltr, Lho, Atr = l1(len_tr), l1(len_ho), l1(abl_tr)
Ctr, Cho = l1(cons_tr), l1(cons_ho)
LtrX, CtrX = ex(len_tr), ex(cons_tr)

# Lenient hold-out rank (within /343) read from the lenient study CSV; conservative rank within /120.
len_csv = pd.read_csv(RESULTS / "triple_barrier_per_instrument.csv")
len_rank = (len_csv[len_csv.pnl_type == "lag1"].sort_values(["instrument", "adj_sharpe_train"],
            ascending=[True, False]).groupby("instrument").head(1).set_index("instrument")["ho_rank"])
cho = cons_hold_grid[cons_hold_grid.pnl_type == "lag1"].copy()
cho["rk"] = cho.groupby("instrument")["adj_sharpe"].rank(ascending=False)

rows = []
for inst in INSTR:
    d = dict(instrument=inst,
             len_pt=Ltr.loc[inst, "pt"], len_sl=Ltr.loc[inst, "sl"], len_h=int(Ltr.loc[inst, "h"]),
             len_pos_rate=Ltr.loc[inst, "pos_rate"], len_n_kept=int(Ltr.loc[inst, "n_kept"]),
             len_vert_frac=Ltr.loc[inst, "vert_frac"], len_adj_tr=Ltr.loc[inst, "adj_sharpe"],
             len_adj_ho=(Lho.loc[inst, "adj_sharpe"] if inst in Lho.index else np.nan),
             len_raw_ho=(Lho.loc[inst, "raw_sharpe"] if inst in Lho.index else np.nan),
             len_exit_adj_tr=(LtrX.loc[inst, "adj_sharpe"] if inst in LtrX.index else np.nan),
             len_ho_rank343=int(len_rank.get(inst, -1)),
             len_vz_adj_tr=(Atr.loc[inst, "adj_sharpe"] if inst in Atr.index else np.nan))
    if inst in Ctr.index:
        rr = cho[(cho.instrument == inst) & (cho.pt == CONS_GEOM[inst][0]) &
                 (cho.sl == CONS_GEOM[inst][1]) & (cho.h == CONS_GEOM[inst][2])]
        d.update(cons_pt=Ctr.loc[inst, "pt"], cons_sl=Ctr.loc[inst, "sl"], cons_h=int(Ctr.loc[inst, "h"]),
                 cons_pos_rate=Ctr.loc[inst, "pos_rate"], cons_n_kept=int(Ctr.loc[inst, "n_kept"]),
                 cons_vert_frac=Ctr.loc[inst, "vert_frac"], cons_adj_tr=Ctr.loc[inst, "adj_sharpe"],
                 cons_adj_ho=(Cho.loc[inst, "adj_sharpe"] if inst in Cho.index else np.nan),
                 cons_raw_ho=(Cho.loc[inst, "raw_sharpe"] if inst in Cho.index else np.nan),
                 cons_n_taken_ho=(int(Cho.loc[inst, "n_taken"]) if inst in Cho.index else 0),
                 cons_exit_adj_tr=(CtrX.loc[inst, "adj_sharpe"] if inst in CtrX.index else np.nan),
                 cons_ho_rank120=(int(rr["rk"].iloc[0]) if len(rr) else -1), floor_ok=True)
    else:
        d["floor_ok"] = False
    rows.append(d)
comp = pd.DataFrame(rows)
comp["geom_changed"] = ~((comp.get("cons_pt") == comp.len_pt) &
                         (comp.get("cons_sl") == comp.len_sl) & (comp.get("cons_h") == comp.len_h))

view = comp[["instrument", "len_pt", "len_sl", "len_h", "cons_pt", "cons_sl", "cons_h",
             "len_pos_rate", "cons_pos_rate", "len_n_kept", "cons_n_kept",
             "len_adj_tr", "cons_adj_tr", "len_adj_ho", "cons_adj_ho", "floor_ok"]]
print("LENIENT vs CONSERVATIVE — per-instrument (lag-1):")
display(view.round(3))
print("\n`vertical_zero` ABLATION (lenient geometry; in-sample lag-1 adj Sharpe) — timeout-rule effect alone:")
display(comp[["instrument", "len_pt", "len_sl", "len_h", "len_adj_tr", "len_vz_adj_tr"]]
        .assign(vz_delta=(comp.len_vz_adj_tr - comp.len_adj_tr)).round(3))
print("\nEXIT-panel (MAXIMALLY circular — label is defined from this return) in-sample adj Sharpe:")
display(comp[["instrument", "len_exit_adj_tr", "cons_exit_adj_tr"]].round(3))

## Section 8 — Honest verdict

The conservative scheme was built to test whether the lenient study's headline adjusted Sharpe (~15
in-sample) was a **labeling artifact**. The honest answer from the numbers above: **mostly not — the
adjusted Sharpe is intrinsic to the oracle construction, not to the lenient geometry.** Four findings.

**(i) Tightening did *not* deflate the in-sample Sharpe.** Median in-sample lag-1 adjusted Sharpe is
essentially unchanged — **lenient 15.37 → conservative 15.86** — with per-instrument deltas mostly within
±1.5 (cl1s 10.5→9.9, es1s 13.1→13.0, gc1s 15.3→15.6, nq1s 15.4→15.9). The `vertical_zero`-only ablation
(lenient geometry, *only* the timeout rule flipped) even **raises** it (median 15.37 → **17.65**; +0 to +6
per name, e.g. ng1s +6.0, ho1s +4.1). So the number is a property of *filtering realized returns by
realized labels*, not of `pt`/`sl`/`h`. **What conservatism genuinely fixes is the mechanism, not the
level:** at `h ≥ 5` the scored lag-1 bar is no longer the *same bar* as the label, so the ~16 Sharpe is now
earned **without** the h=1 single-bar circularity — yet it lands at the level the lenient pick reached
*with* that circularity. That equality is the cleanest evidence the lenient ~15 was **not primarily** an
h=1 overlap artifact.

**(ii) It generalises comparably out-of-sample — no systematic gain or loss.** Median hold-out lag-1
adjusted Sharpe **19.87 → 20.08**. Conservatism helps some names (gc1s 18→24, ng1s 17→26, si1s 24→28, nq1s
24→26) and hurts others (hg1s 24→19, pl1s 22→18, es1s 24→22); the net is a wash. The strict label is
neither more nor less robust OOS at the portfolio median.

**(iii) The cost is selectivity — and the oracle's boundary-seeking behaviour is unchanged.** The
conservative label keeps far fewer trades: median positive rate **0.50 → 0.20** (median vert-frac 0.25).
More tellingly, the search **still pins to the envelope boundary**: **11/11** picks sit at the *tightest
allowed* stop `sl = 0.5`, and **8/11** at the *shortest allowed* horizon `h = 5`. This is the same
behaviour the lenient study showed (it pinned to `sl=0.25`, `h=1`): removing the hair-trigger and the
1-bar horizon **moved the boundary, it did not change the preference.** Only `pt` genuinely entered the
interior (picks span 1.5–3.0, median 2.5). The lenient report's *"tight stop dominates the oracle"* caveat
therefore **survives** conservatism — it is structural, not a grid accident.

**(iv) Thin names still fail.** `ho1s` (~50 events) is unusable under any strict label: it clears the
in-sample floor, but its conservative pick (2.0/0.5/20) yields **0 taken trades on the hold-out** (raw
hold-out Sharpe −31) — the same failure the lenient hold-out exposed, not rescued. It must be excluded
downstream regardless of scheme.

**The exit panel is the tell.** The realized-exit adjusted Sharpe is the **highest** of all (median ~33–36;
ng1s **91**) precisely because the label is *defined from* the exit return — `y=1 ⟺` the exit was a `+pt·σ`
profit-touch, so the kept exit returns are all `≈ +pt·σ` by construction. Its inflation is **mechanical**,
and it is the clearest single demonstration that **every adjusted Sharpe here is an oracle upper bound, not
a backtest.** (The placebo still holds, though — conservative median lag-0 ≈ −0.66 < lag-1 15.86, so the
label encodes genuine *forward* timing, not a generic keep-the-positives trick.)

**Recommendation.** Prefer the **conservative geometry as the meta-model's labeling target** — not because
it scores higher (it does not) but because it is *less circular* (`h ≥ 5` breaks the single-bar overlap),
encodes a *stronger economic prior* (a real `+pt·σ` move, not a hair's-breadth drift), and yields a
*cleaner, more selective* positive class. Use the per-instrument conservative `(pt,sl,h)` for the 10
floor-passing names; **exclude `ho1s`** (structurally too thin). And carry the caveat forward unchanged: the
only honest test is whether a meta-model can **predict** the conservative `y` out-of-sample (out-of-fold
probabilities, CLAUDE.md §6) — the oracle Sharpe, lenient or conservative, is not deployable. Artifacts:
`results/triple_barrier_per_instrument_conservative.csv` and
`results/triple_barrier_lenient_vs_conservative.csv`; **`data/triple_barrier_labels.csv` is left untouched**
(this is an analysis-only study).

In [ ]:
# Verdict inputs — compact quantitative summary the written verdict is built from.
fp = comp[comp.floor_ok]
def med(s): return round(float(np.nanmedian(s.astype(float))), 2)
print("== leniency removed? (in-sample lag-1 adj Sharpe, median over floor-passing) ==")
print(f"   lenient {med(fp.len_adj_tr)}  ->  conservative {med(fp.cons_adj_tr)}   "
      f"(vertical_zero-only ablation on lenient geom: {med(fp.len_vz_adj_tr)})")
print("== generalises? (hold-out lag-1 adj Sharpe, median over floor-passing) ==")
print(f"   lenient {med(fp.len_adj_ho)}  ->  conservative {med(fp.cons_adj_ho)}")
print("== cost? ==")
print(f"   pos-rate median: lenient {med(fp.len_pos_rate)} -> conservative {med(fp.cons_pos_rate)}")
print(f"   vert-frac median: conservative {med(fp.cons_vert_frac)}")
print(f"   floor-FAIL (unusable under conservative): {comp[~comp.floor_ok].instrument.tolist() or 'none'}")
print(f"   conservative picks with <2 taken trades OOS (noisy hold-out): "
      f"{fp[fp.cons_n_taken_ho < 2].instrument.tolist() or 'none'}")
print("== exit-panel (maximally circular: label defined from exit ret) median in-sample adj Sharpe ==")
print(f"   lenient {med(fp.len_exit_adj_tr)}  vs  conservative {med(fp.cons_exit_adj_tr)}")
print("== geometry changed lenient->conservative for: ==")
print("  ", comp[comp.geom_changed].instrument.tolist())

## Section 9 — Artifacts (new files only)

In [ ]:
# (1) Conservative top-3 per (instrument, pnl_type), floor-passing — mirrors the lenient CSV schema + floor cols.
out_rows = []
for lag in ["exit", "lag0", "lag1", "lag2"]:
    out_rows.append(cons_train_vs_holdout(lag).assign(pnl_type=lag))
cons_out = (pd.concat(out_rows)[["instrument", "pnl_type", "pt", "sl", "h", "pos_rate", "n_kept",
                                 "vert_frac", "adj_sharpe_train", "adj_sharpe_hold", "raw_sharpe_hold",
                                 "ho_rank", "n_taken_hold"]]
            .sort_values(["pnl_type", "instrument", "adj_sharpe_train"], ascending=[True, True, False])
            .reset_index(drop=True))
p1 = RESULTS / "triple_barrier_per_instrument_conservative.csv"
cons_out.round(4).to_csv(p1, index=False)
print("wrote", p1, "| rows", len(cons_out))

# (2) The per-instrument lenient-vs-conservative master comparison (lag-1) + exit panel + ablation.
p2 = RESULTS / "triple_barrier_lenient_vs_conservative.csv"
comp.round(4).to_csv(p2, index=False)
print("wrote", p2, "| rows", len(comp), "| floor-pass", int(comp.floor_ok.sum()))
print("\nNOTE: data/triple_barrier_labels.csv is intentionally NOT modified — this is an analysis-only study.")